In [ ]:
from datetime import datetime, date

def get_user_input():
    num_subjects = int(input("How many subjects do you want to add?" ))
    subjects = []

    for _ in range(num_subjects):
        name = input("Enter the subject name:")
        marks = float(input("Enter previous marks or quiz score (%):"))
        practice = float(input("Enter practice completion (%): "))
        revision = float(input("Enter revision completion (%): "))
        last_revision = float(input("Enter last revision completion (%): "))
        topic_comp = float(input("Enter overall topic completion (%):"))
        exam_date_str = input("Enter exam date (YYYY-MM-DD):" ).strip()

        print("Enter 2 or 3 topics for this subject (separated by ,")
        topics_raw = input("Topics: ").split(',')
        topics = [t.strip() for t in topics_raw if t.strip()]

        weak_topic = input(f"Which topic needs the most attention in {name}?")

        focus_topic = weak_topic if weak_topic else (topics[0] if topics else "General Revision")

        subjects.append({
            "name": name,
            "marks": marks,
            "practice": practice,
            "revision": revision,
            "last_revision":last_revision,
            "topic": topic_comp,
            "exam_date": exam_date_str,
            "topics": topics,
            "focus_topic": focus_topic
        })
    return subjects

def calculate_priority(subject):
    try:
        exam_date = datetime.strptime(subject["exam_date"], "%Y-%m-%d").date()
        days_left = (exam_date - date.today()).days
        days_left = max(days_left, 1)
    except ValueError:
        days_left = 30

    
    marks_deficit = (100 - subject["marks"]) * 0.15
    practice_deficit = (100 - subject["practice"]) * 0.25
    revision_deficit = (100 - subject["revision"]) * 0.25
    last_revision_deficit = (100 - subject["last_revision"]) * 0.20
    topic_deficit = (100 - subject["topic"]) * 0.15

    total_deficit = marks_deficit + practice_deficit + revision_deficit + last_revision_deficit + topic_deficit

    # Urgency based on days left, higher urgency for closer exams
    urgency = max(1.0, 20.0 / days_left)
    score = total_deficit * urgency

    # Determine priority level and suggested study time
    if score >= 55:
        priority = "HIGH"
        study_time = "2 hours"
    elif score >= 33:
        priority = "MEDIUM"
        study_time = "1 hour"
    else:
        priority = "LOW"
        study_time = "30 minutes"

    
    metrics = {
        "Low_marks": (100 - subject["marks"]) * 0.15,
        "Practice_incomplete": (100 - subject["practice"]) * 0.25,
        "Low_revision": (100 - subject["revision"]) * 0.25,
        "Last_revision_incomplete": (100 - subject["last_revision"]) * 0.20,
        "Topic completion pending": (100 - subject["topic"]) * 0.15
    }
    highest_metric_name = max(metrics, key=metrics.get)

    
    if days_left <= 7:
        reason = f"{highest_metric_name} + upcoming exam ({days_left} days left)"
    elif subject["marks"] > 80 and subject["revision"] > 80:
        reason = "Good marks + high revision"
    else:
        reason = highest_metric_name

    return {
        "name": subject["name"],
        "priority": priority,
        "required_study": study_time,
        "focus": subject["focus_topic"],
        "reason": reason,
        "score": score
    }

def display_today_priority(subjects):
    results = [calculate_priority(s) for s in subjects]

    
    priority_order = {"HIGH": 3, "MEDIUM": 2, "LOW": 1}
    results.sort(key=lambda x: (priority_order[x["priority"]], x["score"]), reverse=True)

    print("PRIORITY for today")

    for item in results:
        print(f"Subject: {item['name']}")
        print(f"Priority: {item['priority']}")
        print(f"Required study: {item['required_study']}")
        print(f"Focus topic: {item['focus']}")
        print(f"Reason: {item['reason']}")

if __name__ == '__main__':
    user_subjects = get_user_input()
    display_today_priority(user_subjects)

Enter 2 or 3 topics for this subject (separated by ,
TODAY'S PRIORITY
Subject: maths
Priority: MEDIUM
Required study: 1 hour
Focus topic: add
Reason: Topic completion pending
